# Gomoku Consultant Policy-Value Training on Kaggle

This notebook trains a lightweight supervised CNN advisor from Gomoku/Caro self-play JSONL data.

Report framing: top-1 measures agreement with the self-play engine label, not objectively optimal Gomoku play. Top-3 and top-5 are reported because many positions have multiple reasonable moves.

## Expected Kaggle Setup

1. Push or upload the project data folder as a Kaggle Dataset.
2. Attach that dataset to this notebook.
3. Enable GPU if available.
4. Run all cells.

The notebook auto-discovers `*.jsonl` files under `/kaggle/input`. It expects rows with `board`, `prob`, and `reward` fields.

In [ ]:
import json
import math
import os
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, IterableDataset

print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())

In [ ]:
class CFG:
    BOARD_SIZE = 15
    NUM_ACTIONS = BOARD_SIZE * BOARD_SIZE
    SEED = 42
    VAL_RATIO = 0.10
    TEST_RATIO = 0.10
    MAX_TRAIN_SAMPLES = 200_000
    MAX_EVAL_SAMPLES = 50_000
    BATCH_SIZE = 256
    EPOCHS = 5
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    VALUE_WEIGHT = 0.25
    NUM_WORKERS = 0
    OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('kaggle_working')
    CHECKPOINT_PATH = OUTPUT_DIR / 'consultant_model.pt'
    METRICS_PATH = OUTPUT_DIR / 'metrics.json'
    SPLIT_PATH = OUTPUT_DIR / 'split_manifest.json'

random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Discover Dataset Files

Kaggle usually mounts input datasets under `/kaggle/input/<dataset-name>/...`. The code below recursively finds JSONL files.

In [ ]:
def find_jsonl_files():
    roots = [Path('/kaggle/input'), Path('data')]
    files = []
    for root in roots:
        if root.exists():
            files.extend(sorted(root.rglob('*.jsonl')))
    return sorted({p.resolve() for p in files if p.is_file()})

all_files = find_jsonl_files()
print('jsonl files:', len(all_files))
print('first files:')
for path in all_files[:10]:
    print(' ', path)

if not all_files:
    raise FileNotFoundError('No JSONL files found. Attach the Kaggle dataset that contains data/*.jsonl.')

## Inspect JSONL Schema

This is a quick sanity check. It samples a small number of rows without loading the full dataset into memory.

In [ ]:
def inspect_schema(files, max_files=5, max_lines_per_file=200):
    keys = Counter()
    rewards = Counter()
    stages = Counter()
    stats = Counter()
    examples = []
    for path in files[:max_files]:
        with path.open('r', encoding='utf-8') as f:
            for line_no, line in enumerate(f, 1):
                if line_no > max_lines_per_file:
                    break
                obj = json.loads(line)
                stats['rows'] += 1
                keys.update(obj.keys())
                rewards[obj.get('reward')] += 1
                if 'stage' in obj:
                    stages[obj.get('stage')] += 1
                board = obj.get('board')
                prob = obj.get('prob')
                if isinstance(board, list) and len(board) == CFG.BOARD_SIZE and all(isinstance(r, list) and len(r) == CFG.BOARD_SIZE for r in board):
                    stats['valid_board'] += 1
                if isinstance(prob, list) and len(prob) == CFG.NUM_ACTIONS:
                    stats['valid_prob'] += 1
                    nz = [i for i, v in enumerate(prob) if v != 0]
                    if len(nz) == 1 and prob[nz[0]] == 1.0:
                        stats['one_hot_prob'] += 1
                    elif len(nz) == 0:
                        stats['zero_prob'] += 1
                if len(examples) < 2:
                    examples.append({k: obj[k] for k in obj.keys() if k != 'board' and k != 'prob'})
    return {
        'stats': dict(stats),
        'keys': dict(keys),
        'rewards': dict(rewards),
        'stages': dict(stages),
        'examples_without_large_fields': examples,
    }

schema_report = inspect_schema(all_files)
schema_report

## File-Level Train/Validation/Test Split

The split is by JSONL file, not by row, to reduce leakage from similar positions in the same source shard.

In [ ]:
def split_files(files):
    files = list(files)
    rng = random.Random(CFG.SEED)
    rng.shuffle(files)
    n = len(files)
    n_test = max(1, int(round(n * CFG.TEST_RATIO)))
    n_val = max(1, int(round(n * CFG.VAL_RATIO)))
    test_files = files[:n_test]
    val_files = files[n_test:n_test + n_val]
    train_files = files[n_test + n_val:]
    return train_files, val_files, test_files

train_files, val_files, test_files = split_files(all_files)
split_manifest = {
    'seed': CFG.SEED,
    'train_files': [str(p) for p in train_files],
    'val_files': [str(p) for p in val_files],
    'test_files': [str(p) for p in test_files],
}
CFG.SPLIT_PATH.write_text(json.dumps(split_manifest, indent=2), encoding='utf-8')
print('train/val/test files:', len(train_files), len(val_files), len(test_files))
print('split saved to', CFG.SPLIT_PATH)

## Dataset and Board Encoding

Input tensor shape is `(3, 15, 15)`:

- Channel 0: stones with value `1`
- Channel 1: stones with value `-1`
- Channel 2: empty squares

Rows where `prob` is all zeros are skipped for policy training/evaluation because they do not contain a move label.

In [ ]:
def encode_board(board_arr):
    x = np.zeros((3, CFG.BOARD_SIZE, CFG.BOARD_SIZE), dtype=np.float32)
    x[0] = (board_arr == 1)
    x[1] = (board_arr == -1)
    x[2] = (board_arr == 0)
    return x

def extract_target(prob):
    if not isinstance(prob, list) or len(prob) != CFG.NUM_ACTIONS:
        return None
    arr = np.asarray(prob, dtype=np.float32)
    if not np.isfinite(arr).all() or float(arr.sum()) <= 0.0:
        return None
    return int(arr.argmax())

def parse_reward(value):
    try:
        return float(np.clip(float(value), -1.0, 1.0))
    except (TypeError, ValueError):
        return 0.0

def augment_board_and_target(board_arr, target_idx):
    target_mask = np.zeros((CFG.BOARD_SIZE, CFG.BOARD_SIZE), dtype=np.uint8)
    target_mask.flat[target_idx] = 1
    k = random.randint(0, 3)
    board_arr = np.rot90(board_arr, k)
    target_mask = np.rot90(target_mask, k)
    if random.random() < 0.5:
        board_arr = np.fliplr(board_arr)
        target_mask = np.fliplr(target_mask)
    if random.random() < 0.5:
        board_arr = np.flipud(board_arr)
        target_mask = np.flipud(target_mask)
    return np.ascontiguousarray(board_arr), int(target_mask.argmax())

class JsonlGomokuDataset(IterableDataset):
    def __init__(self, files, max_samples=None, augment=False, shuffle_files=False, seed=42):
        self.files = list(files)
        self.max_samples = max_samples
        self.augment = augment
        self.shuffle_files = shuffle_files
        self.seed = seed

    def __iter__(self):
        files = list(self.files)
        if self.shuffle_files:
            random.Random(self.seed).shuffle(files)
        emitted = 0
        for path in files:
            with Path(path).open('r', encoding='utf-8') as f:
                for line in f:
                    if self.max_samples is not None and emitted >= self.max_samples:
                        return
                    try:
                        obj = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    board = obj.get('board')
                    if not (isinstance(board, list) and len(board) == CFG.BOARD_SIZE):
                        continue
                    board_arr = np.asarray(board, dtype=np.int8)
                    if board_arr.shape != (CFG.BOARD_SIZE, CFG.BOARD_SIZE):
                        continue
                    target_idx = extract_target(obj.get('prob'))
                    if target_idx is None:
                        continue
                    row, col = divmod(target_idx, CFG.BOARD_SIZE)
                    if board_arr[row, col] != 0:
                        continue
                    if self.augment:
                        board_arr, target_idx = augment_board_and_target(board_arr, target_idx)
                    occupied = (board_arr.reshape(-1) != 0)
                    x = encode_board(board_arr)
                    reward = parse_reward(obj.get('reward'))
                    emitted += 1
                    yield (
                        torch.from_numpy(x),
                        torch.tensor(target_idx, dtype=torch.long),
                        torch.tensor(reward, dtype=torch.float32),
                        torch.from_numpy(occupied.astype(np.bool_)),
                    )

In [ ]:
train_ds = JsonlGomokuDataset(train_files, max_samples=CFG.MAX_TRAIN_SAMPLES, augment=True, shuffle_files=True, seed=CFG.SEED)
val_ds = JsonlGomokuDataset(val_files, max_samples=CFG.MAX_EVAL_SAMPLES, augment=False)
test_ds = JsonlGomokuDataset(test_files, max_samples=CFG.MAX_EVAL_SAMPLES, augment=False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS)

batch = next(iter(DataLoader(JsonlGomokuDataset(train_files, max_samples=4), batch_size=4)))
print('x', batch[0].shape, 'target', batch[1].shape, 'value', batch[2].shape, 'occupied', batch[3].shape)

## Policy-Value CNN

The model is intentionally compact so it can run as a real-time advisor.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class PolicyValueNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 128),
        )
        self.policy_head = nn.Sequential(
            nn.Conv2d(128, 2, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(2 * CFG.NUM_ACTIONS, CFG.NUM_ACTIONS),
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(128, 1, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(CFG.NUM_ACTIONS, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        h = self.trunk(x)
        policy_logits = self.policy_head(h)
        value = self.value_head(h).squeeze(-1)
        return policy_logits, value

model = PolicyValueNet().to(device)
param_count = sum(p.numel() for p in model.parameters())
print('parameters:', f'{param_count:,}')

## Metrics and Baselines

Evaluation masks occupied squares before sorting top-K moves. This keeps advisor recommendations legal.

In [ ]:
def mask_illegal_logits(logits, occupied):
    return logits.masked_fill(occupied.to(dtype=torch.bool, device=logits.device), -1e9)

def batch_topk_stats(logits, targets, occupied, ks=(1, 3, 5)):
    targets = targets.to(logits.device)
    occupied = occupied.to(logits.device)
    raw_top1 = logits.argmax(dim=1)
    illegal_before = occupied.gather(1, raw_top1.view(-1, 1)).float().sum().item()
    masked = mask_illegal_logits(logits, occupied)
    max_k = max(ks)
    topk = masked.topk(max_k, dim=1).indices
    out = {'n': targets.numel(), 'illegal_before': illegal_before}
    for k in ks:
        out[f'top{k}'] = (topk[:, :k] == targets.view(-1, 1)).any(dim=1).float().sum().item()
    return out

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    totals = Counter()
    value_abs_error = 0.0
    value_sq_error = 0.0
    policy_loss_sum = 0.0
    value_loss_sum = 0.0
    ce = nn.CrossEntropyLoss(reduction='sum')
    mse = nn.MSELoss(reduction='sum')
    for x, targets, values, occupied in loader:
        x = x.to(device)
        targets = targets.to(device)
        values = values.to(device)
        occupied = occupied.to(device)
        logits, pred_values = model(x)
        masked_logits = mask_illegal_logits(logits, occupied)
        stats = batch_topk_stats(logits, targets, occupied)
        totals.update(stats)
        policy_loss_sum += ce(masked_logits, targets).item()
        value_loss_sum += mse(pred_values, values).item()
        value_abs_error += (pred_values - values).abs().sum().item()
        value_sq_error += ((pred_values - values) ** 2).sum().item()
    n = max(1, totals['n'])
    return {
        'samples': int(n),
        'top1_accuracy': totals['top1'] / n,
        'top3_accuracy': totals['top3'] / n,
        'top5_accuracy': totals['top5'] / n,
        'illegal_top1_rate_before_mask': totals['illegal_before'] / n,
        'illegal_top1_rate_after_mask': 0.0,
        'policy_loss': policy_loss_sum / n,
        'value_loss': value_loss_sum / n,
        'value_mae': value_abs_error / n,
        'value_mse': value_sq_error / n,
    }

def legal_indices_from_occupied(occupied_row):
    occupied_np = occupied_row.cpu().numpy().astype(bool)
    return np.flatnonzero(~occupied_np)

def center_order_for_occupied(occupied_row):
    legal = legal_indices_from_occupied(occupied_row)
    center = (CFG.BOARD_SIZE - 1) / 2
    return sorted(legal.tolist(), key=lambda idx: ((idx // CFG.BOARD_SIZE - center) ** 2, (idx % CFG.BOARD_SIZE - center) ** 2))

def evaluate_baselines(loader):
    totals = Counter()
    for _, targets, _, occupied in loader:
        for target, occ in zip(targets, occupied):
            legal = legal_indices_from_occupied(occ)
            if len(legal) == 0:
                continue
            target = int(target.item())
            totals['n'] += 1
            totals['random_top1'] += 1.0 / len(legal)
            totals['random_top3'] += min(3, len(legal)) / len(legal)
            totals['random_top5'] += min(5, len(legal)) / len(legal)
            center_ranked = center_order_for_occupied(occ)
            totals['center_top1'] += float(target in center_ranked[:1])
            totals['center_top3'] += float(target in center_ranked[:3])
            totals['center_top5'] += float(target in center_ranked[:5])
    n = max(1, totals['n'])
    return {
        'random_legal': {'top1': totals['random_top1'] / n, 'top3': totals['random_top3'] / n, 'top5': totals['random_top5'] / n},
        'center_first': {'top1': totals['center_top1'] / n, 'top3': totals['center_top3'] / n, 'top5': totals['center_top5'] / n},
    }

## Training Loop

The best checkpoint is selected by validation top-1 agreement. You can increase `MAX_TRAIN_SAMPLES` and `EPOCHS` for a stronger report run.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()

history = []
best_val_top1 = -1.0

for epoch in range(1, CFG.EPOCHS + 1):
    model.train()
    running = Counter()
    t0 = time.perf_counter()
    train_loader = DataLoader(
        JsonlGomokuDataset(train_files, max_samples=CFG.MAX_TRAIN_SAMPLES, augment=True, shuffle_files=True, seed=CFG.SEED + epoch),
        batch_size=CFG.BATCH_SIZE,
        num_workers=CFG.NUM_WORKERS,
    )
    for x, targets, values, occupied in train_loader:
        x = x.to(device)
        targets = targets.to(device)
        values = values.to(device)
        occupied = occupied.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits, pred_values = model(x)
        masked_logits = mask_illegal_logits(logits, occupied)
        policy_loss = ce_loss(masked_logits, targets)
        value_loss = mse_loss(pred_values, values)
        loss = policy_loss + CFG.VALUE_WEIGHT * value_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
        optimizer.step()
        batch_n = targets.numel()
        running['samples'] += batch_n
        running['loss_sum'] += float(loss.item()) * batch_n
        running['policy_loss_sum'] += float(policy_loss.item()) * batch_n
        running['value_loss_sum'] += float(value_loss.item()) * batch_n
    val_loader = DataLoader(JsonlGomokuDataset(val_files, max_samples=CFG.MAX_EVAL_SAMPLES), batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS)
    val_metrics = evaluate_model(model, val_loader)
    elapsed = time.perf_counter() - t0
    train_samples = max(1, running['samples'])
    epoch_metrics = {
        'epoch': epoch,
        'elapsed_seconds': elapsed,
        'train_samples': int(running['samples']),
        'train_loss': running['loss_sum'] / train_samples,
        'train_policy_loss': running['policy_loss_sum'] / train_samples,
        'train_value_loss': running['value_loss_sum'] / train_samples,
        'val': val_metrics,
    }
    history.append(epoch_metrics)
    print(json.dumps(epoch_metrics, indent=2))
    if val_metrics['top1_accuracy'] > best_val_top1:
        best_val_top1 = val_metrics['top1_accuracy']
        torch.save({'model_state_dict': model.state_dict(), 'cfg': {k: v for k, v in CFG.__dict__.items() if k.isupper() and isinstance(v, (int, float, str, bool))}}, CFG.CHECKPOINT_PATH)
        print('saved best checkpoint:', CFG.CHECKPOINT_PATH)

## Final Test Evaluation

These are the numbers to use in the report table.

In [ ]:
checkpoint = torch.load(CFG.CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

test_loader = DataLoader(JsonlGomokuDataset(test_files, max_samples=CFG.MAX_EVAL_SAMPLES), batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS)
test_metrics = evaluate_model(model, test_loader)
baseline_loader = DataLoader(JsonlGomokuDataset(test_files, max_samples=CFG.MAX_EVAL_SAMPLES), batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS)
baseline_metrics = evaluate_baselines(baseline_loader)

final_metrics = {
    'config': {
        'seed': CFG.SEED,
        'max_train_samples': CFG.MAX_TRAIN_SAMPLES,
        'max_eval_samples': CFG.MAX_EVAL_SAMPLES,
        'batch_size': CFG.BATCH_SIZE,
        'epochs': CFG.EPOCHS,
        'lr': CFG.LR,
        'value_weight': CFG.VALUE_WEIGHT,
        'device': str(device),
    },
    'history': history,
    'test': test_metrics,
    'baselines': baseline_metrics,
}
print(json.dumps(final_metrics['test'], indent=2))
print(json.dumps(final_metrics['baselines'], indent=2))

## Inference Latency

Mean latency is measured on repeated single-board inference. This helps justify real-time advisor usage.

In [ ]:
@torch.no_grad()
def measure_latency(model, repeats=200):
    model.eval()
    sample_loader = DataLoader(JsonlGomokuDataset(test_files, max_samples=1), batch_size=1, num_workers=0)
    x, _, _, occupied = next(iter(sample_loader))
    x = x.to(device)
    occupied = occupied.to(device)
    for _ in range(10):
        logits, value = model(x)
        _ = mask_illegal_logits(logits, occupied).softmax(dim=1)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        logits, value = model(x)
        _ = mask_illegal_logits(logits, occupied).softmax(dim=1)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return {'latency_ms_mean': float(np.mean(times)), 'latency_ms_p95': float(np.percentile(times, 95))}

latency_metrics = measure_latency(model)
final_metrics['latency'] = latency_metrics
latency_metrics

## Tactical Diagnostics

This lightweight diagnostic checks whether a trained advisor ranks obvious tactical moves highly. It is not a proof of engine strength.

In [ ]:
TACTICAL_CASES = [
    {
        'name': 'ai_win_horizontal',
        'board': [
            '...............', '...............', '...............', '...............', '...............',
            '...............', '...............', '.....OOOO......', '...............', '...............',
            '...............', '...............', '...............', '...............', '...............'
        ],
        'expected_moves': [(7, 4), (7, 9)],
        'tags': ['win', 'open_four'],
    },
    {
        'name': 'block_human_horizontal',
        'board': [
            '...............', '...............', '...............', '...............', '...............',
            '...............', '...............', '.....XXXX......', '...............', '...............',
            '...............', '...............', '...............', '...............', '...............'
        ],
        'expected_moves': [(7, 4), (7, 9)],
        'tags': ['block', 'open_four'],
    },
]

def board_from_strings(rows):
    mapping = {'.': 0, 'O': 1, 'X': -1}
    return np.asarray([[mapping[ch] for ch in row] for row in rows], dtype=np.int8)

@torch.no_grad()
def predict_top_moves(board_arr, top_k=5):
    occupied = torch.from_numpy((board_arr.reshape(-1) != 0).astype(np.bool_)).unsqueeze(0).to(device)
    x = torch.from_numpy(encode_board(board_arr)).unsqueeze(0).to(device)
    logits, value = model(x)
    probs = mask_illegal_logits(logits, occupied).softmax(dim=1)[0]
    top = probs.topk(top_k).indices.cpu().numpy().tolist()
    return [(idx // CFG.BOARD_SIZE, idx % CFG.BOARD_SIZE, float(probs[idx].cpu())) for idx in top], float(value.item())

tactical_results = []
for case in TACTICAL_CASES:
    board_arr = board_from_strings(case['board'])
    top_moves, value = predict_top_moves(board_arr, top_k=5)
    expected = set(tuple(m) for m in case['expected_moves'])
    top_coords = [(r, c) for r, c, _ in top_moves]
    tactical_results.append({
        'name': case['name'],
        'tags': case['tags'],
        'expected_moves': case['expected_moves'],
        'top_moves': top_moves,
        'top1_hit': top_coords[0] in expected,
        'top3_hit': any(move in expected for move in top_coords[:3]),
        'value': value,
    })

final_metrics['tactical'] = tactical_results
tactical_results

## Export Artifacts

Download these files from Kaggle output after the run:

- `consultant_model.pt`
- `metrics.json`
- `split_manifest.json`
- `report_table.md`

In [ ]:
CFG.METRICS_PATH.write_text(json.dumps(final_metrics, indent=2), encoding='utf-8')

report_table = f"""| Model | Top-1 | Top-3 | Top-5 | Illegal Top-1 | Value MAE | Mean Latency ms |
|---|---:|---:|---:|---:|---:|---:|
| Random legal | {baseline_metrics['random_legal']['top1']:.4f} | {baseline_metrics['random_legal']['top3']:.4f} | {baseline_metrics['random_legal']['top5']:.4f} | 0.0000 | - | - |
| Center-first | {baseline_metrics['center_first']['top1']:.4f} | {baseline_metrics['center_first']['top3']:.4f} | {baseline_metrics['center_first']['top5']:.4f} | 0.0000 | - | - |
| Consultant CNN | {test_metrics['top1_accuracy']:.4f} | {test_metrics['top3_accuracy']:.4f} | {test_metrics['top5_accuracy']:.4f} | {test_metrics['illegal_top1_rate_after_mask']:.4f} | {test_metrics['value_mae']:.4f} | {latency_metrics['latency_ms_mean']:.2f} |
"""
report_table_path = CFG.OUTPUT_DIR / 'report_table.md'
report_table_path.write_text(report_table, encoding='utf-8')
print(report_table)
print('checkpoint:', CFG.CHECKPOINT_PATH)
print('metrics:', CFG.METRICS_PATH)
print('split:', CFG.SPLIT_PATH)
print('report table:', report_table_path)